# Astrometry centering diagnostics: centroid definition drives the large offsets

**Goal**: Test what actually causes the 40-50 mas astrometry residual before changing the solver.

The working hypothesis is now explicit: the large per-source offset is mostly **centering / centroid-definition scatter**, not a smooth concordance-field error.

What we inspect:
1. **Centering definition** - how much the measured VIS-Rubin offset changes when we use detection, flux-weighted, or PSF-fit centroids
2. **Centroiding quality** - VIS and Rubin cutouts with centroid overlays
3. **Match quality** - offset vector field per tile
4. **Offset vs SNR** - whether residuals follow the expected centroid-noise scale
5. **Morphology and band dependence** - whether galaxies or r-vs-i centroid changes explain the tail
6. **Residuals after bulk field removal** - what is left after subtracting the smooth field a concordance file could model


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Ellipse
from matplotlib.colors import LogNorm
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import astropy.units as u
from scipy.ndimage import gaussian_filter, maximum_filter
from scipy.optimize import least_squares

plt.rcParams['figure.dpi'] = 120
plt.rcParams['image.origin'] = 'lower'

def find_repo_root(start: Path = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for cand in [start, *start.parents]:
        if (cand / 'models').exists() and (cand / 'data').exists():
            return cand
    return start

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT / 'models'))

from astrometry2.source_matching import (
    detect_sources, match_sources_wcs, refine_centroids_psf_fit,
    refine_centroids_in_band, build_detection_image, robust_sigma,
    safe_header_from_card_string, estimate_source_snr, RUBIN_BAND_ORDER,
)
from astrometry2.dataset import _to_float32, _normalize_patch

# ── Data paths ───────────────────────────────────────────────────────
RUBIN_DIR = ROOT / 'data' / 'rubin_tiles_all'
EUCLID_DIR = ROOT / 'data' / 'euclid_tiles_all'

rubin_files = sorted(RUBIN_DIR.glob('tile_x*_y*.npz'))
euclid_files = sorted(EUCLID_DIR.glob('tile_x*_y*_euclid.npz'))
print(f'Rubin tiles: {len(rubin_files)},  Euclid tiles: {len(euclid_files)}')

In [ ]:
# ── Helper: load a tile pair and run the full detection + matching + PSF-fit pipeline ──

def load_tile(rubin_path, euclid_path):
    """Load a tile pair and return all images, WCS, and variance."""
    rdata = np.load(rubin_path, allow_pickle=True)
    edata = np.load(euclid_path, allow_pickle=True)

    rubin_cube = rdata['img']   # [6, 512, 512]
    rubin_var = rdata['var'] if 'var' in rdata else None
    rwcs = WCS(rdata['wcs_hdr'].item())
    tile_id = str(rdata['tile_id'])

    vis_img = np.nan_to_num(_to_float32(edata['img_VIS']), nan=0.0)
    vis_var = _to_float32(edata['var_VIS']) if 'var_VIS' in edata else None
    vhdr = safe_header_from_card_string(edata['wcs_VIS'].item())
    vwcs = WCS(vhdr)

    return {
        'tile_id': tile_id,
        'rubin_cube': rubin_cube, 'rubin_var': rubin_var, 'rwcs': rwcs,
        'vis_img': vis_img, 'vis_var': vis_var, 'vwcs': vwcs,
        'edata': edata,
    }


def run_matching_pipeline(tile, target_band='r', detect_bands=('g', 'r', 'i', 'z')):
    """Run full detection → matching → PSF-fit centroiding on one tile.
    
    Returns a dict with matched source info including PSF-fit centroids and SNR.
    """
    rubin_cube = tile['rubin_cube']
    vis_img = tile['vis_img']
    rwcs, vwcs = tile['rwcs'], tile['vwcs']

    detect_bands_full = [f'rubin_{b}' for b in detect_bands]
    rubin_det = build_detection_image(rubin_cube, detect_bands_full)
    target_idx = RUBIN_BAND_ORDER.index(target_band)
    rubin_target = np.nan_to_num(_to_float32(rubin_cube[target_idx]), nan=0.0)

    # Detect sources
    rx, ry = detect_sources(rubin_det, nsig=4.5, smooth_sigma=1.0, min_dist=7, max_sources=600)
    vx, vy = detect_sources(vis_img,   nsig=4.0, smooth_sigma=1.2, min_dist=9, max_sources=800)

    # Match
    matched = match_sources_wcs(rx, ry, vx, vy, rwcs, vwcs,
                                max_sep_arcsec=0.12, clip_sigma=3.5, max_matches=256)
    if matched['vis_xy'].shape[0] == 0:
        return None

    # PSF-fit centroids in Rubin target band
    rubin_refined, rubin_snr, rubin_csig = refine_centroids_psf_fit(
        rubin_target, matched['rubin_xy'], radius=5, fwhm_guess=3.0)

    # PSF-fit centroids in VIS
    vis_refined, vis_snr, vis_csig = refine_centroids_psf_fit(
        vis_img, matched['vis_xy'], radius=5, fwhm_guess=4.0)

    # Simple flux-weighted centroids for comparison
    rubin_flux = refine_centroids_in_band(rubin_target, matched['rubin_xy'], radius=3)
    vis_flux = refine_centroids_in_band(vis_img, matched['vis_xy'], radius=3)

    # Compute sky offsets using PSF-fit centroids
    r_ra, r_dec = rwcs.wcs_pix2world(rubin_refined[:, 0], rubin_refined[:, 1], 0)
    v_ra, v_dec = vwcs.wcs_pix2world(vis_refined[:, 0], vis_refined[:, 1], 0)
    dra = (v_ra - r_ra) * np.cos(np.deg2rad(v_dec)) * 3600.0   # arcsec
    ddec = (v_dec - r_dec) * 3600.0

    # Same but with flux-weighted centroids
    r_ra_f, r_dec_f = rwcs.wcs_pix2world(rubin_flux[:, 0], rubin_flux[:, 1], 0)
    v_ra_f, v_dec_f = vwcs.wcs_pix2world(vis_flux[:, 0], vis_flux[:, 1], 0)
    dra_flux = (v_ra_f - r_ra_f) * np.cos(np.deg2rad(v_dec_f)) * 3600.0
    ddec_flux = (v_dec_f - r_dec_f) * 3600.0

    # Aperture SNR for VIS (independent of PSF fit success)
    vis_snr_aper = estimate_source_snr(vis_img, matched['vis_xy'], aperture_radius=5)

    return {
        'n_rubin_det': len(rx), 'n_vis_det': len(vx), 'n_matched': matched['vis_xy'].shape[0],
        # Original detection positions
        'rubin_det_xy': matched['rubin_xy'],
        'vis_det_xy': matched['vis_xy'],
        # PSF-fit refined positions
        'rubin_psf_xy': rubin_refined, 'vis_psf_xy': vis_refined,
        'rubin_snr': rubin_snr, 'vis_snr': vis_snr,
        'rubin_csig_px': rubin_csig, 'vis_csig_px': vis_csig,
        # Flux-weighted refined positions
        'rubin_flux_xy': rubin_flux, 'vis_flux_xy': vis_flux,
        # Sky offsets (PSF-fit)
        'dra': dra, 'ddec': ddec,
        'offset_mag': np.hypot(dra, ddec) * 1000,  # mas
        # Sky offsets (flux-weighted)
        'dra_flux': dra_flux, 'ddec_flux': ddec_flux,
        'offset_mag_flux': np.hypot(dra_flux, ddec_flux) * 1000,
        # Combined SNR
        'vis_snr_aper': vis_snr_aper,
        # Target band image for cutouts
        'rubin_target': rubin_target,
        'target_band': target_band,
    }

print('Pipeline functions ready.')

## 1. Run pipeline on a sample of tiles

Process ~20 tiles spread across the mosaic to get representative diagnostics.

In [ ]:
# Build tile pair list
pair_map = {}
for rp in rubin_files:
    tid = rp.stem
    ep = EUCLID_DIR / f'{tid}_euclid.npz'
    if ep.exists():
        pair_map[tid] = (rp, ep)

print(f'{len(pair_map)} matched tile pairs')

# Sample ~20 tiles evenly spaced through the list
all_tids = sorted(pair_map.keys())
N_SAMPLE = 20
step = max(1, len(all_tids) // N_SAMPLE)
sample_tids = all_tids[::step][:N_SAMPLE]
print(f'Sampling {len(sample_tids)} tiles for diagnostics')

In [ ]:
# Run the pipeline on all sample tiles — takes ~1-2 min
results = {}
for tid in sample_tids:
    rp, ep = pair_map[tid]
    try:
        tile = load_tile(rp, ep)
        m = run_matching_pipeline(tile, target_band='r')
        if m is not None and m['n_matched'] >= 10:
            m['tile'] = tile  # keep tile data for cutout plots
            results[tid] = m
            print(f'  {tid}: {m["n_rubin_det"]} R det, {m["n_vis_det"]} V det, '
                  f'{m["n_matched"]} matched, median |off|={np.median(m["offset_mag"]):.1f} mas')
    except Exception as exc:
        print(f'  {tid}: FAILED ({exc})')

print(f'\n{len(results)} tiles processed successfully')

## 2. Centering diagnostic - does the offset move with centroid definition?

This is the direct test for the centering hypothesis. For each matched source we recompute the sky offset three ways:
- detection centroids from the matcher,
- simple flux-weighted recentered positions,
- PSF-fit recentered positions.

Then we measure how far each instrument's centroid moves from the detection seed to the PSF-fit center. If that centering term is comparable to the measured astrometry offset, the dominant error is source-level centroid definition, not a smooth concordance field.


In [ ]:
def sky_offset_vector(wcs_a, xy_a, wcs_b, xy_b):
    """Return sky vector (b - a) in arcsec for two WCS/xy arrays."""
    ra_a, dec_a = wcs_a.wcs_pix2world(xy_a[:, 0], xy_a[:, 1], 0)
    ra_b, dec_b = wcs_b.wcs_pix2world(xy_b[:, 0], xy_b[:, 1], 0)
    dra = (ra_b - ra_a) * np.cos(np.deg2rad(dec_b)) * 3600.0
    ddec = (dec_b - dec_a) * 3600.0
    return np.stack([dra, ddec], axis=1)


def vector_mag_mas(vec_arcsec):
    return np.hypot(vec_arcsec[:, 0], vec_arcsec[:, 1]) * 1000.0


# Pool source-by-source offset definitions across the same processed tiles.
det_offset_vecs = []
flux_offset_vecs = []
psf_offset_vecs = []
psf_resid_vecs = []
rubin_center_shift_vecs = []
vis_center_shift_vecs = []
centering_delta_vecs = []
bulk_mags_by_tile = []

for tid, m in results.items():
    tile = m['tile']
    rwcs, vwcs = tile['rwcs'], tile['vwcs']

    det_vec = sky_offset_vector(rwcs, m['rubin_det_xy'], vwcs, m['vis_det_xy'])
    flux_vec = np.stack([m['dra_flux'], m['ddec_flux']], axis=1)
    psf_vec = np.stack([m['dra'], m['ddec']], axis=1)

    # Same-instrument recentering vectors: PSF-fit center minus detection seed.
    rub_shift = sky_offset_vector(rwcs, m['rubin_det_xy'], rwcs, m['rubin_psf_xy'])
    vis_shift = sky_offset_vector(vwcs, m['vis_det_xy'], vwcs, m['vis_psf_xy'])

    # This is the actual change in the measured VIS-Rubin offset induced by recentering.
    centering_delta = psf_vec - det_vec

    bulk_vec = np.array([np.median(psf_vec[:, 0]), np.median(psf_vec[:, 1])])
    psf_resid_vec = psf_vec - bulk_vec[None, :]
    bulk_mags_by_tile.append(float(np.hypot(*bulk_vec) * 1000.0))

    det_offset_vecs.append(det_vec)
    flux_offset_vecs.append(flux_vec)
    psf_offset_vecs.append(psf_vec)
    psf_resid_vecs.append(psf_resid_vec)
    rubin_center_shift_vecs.append(rub_shift)
    vis_center_shift_vecs.append(vis_shift)
    centering_delta_vecs.append(centering_delta)


det_offset_vec_all = np.concatenate(det_offset_vecs)
flux_offset_vec_all = np.concatenate(flux_offset_vecs)
psf_offset_vec_all = np.concatenate(psf_offset_vecs)
psf_resid_vec_all = np.concatenate(psf_resid_vecs)
rubin_center_shift_vec_all = np.concatenate(rubin_center_shift_vecs)
vis_center_shift_vec_all = np.concatenate(vis_center_shift_vecs)
centering_delta_vec_all = np.concatenate(centering_delta_vecs)

# Magnitudes in mas.
det_offset_mag_all = vector_mag_mas(det_offset_vec_all)
flux_offset_mag_all = vector_mag_mas(flux_offset_vec_all)
psf_offset_mag_all = vector_mag_mas(psf_offset_vec_all)
psf_resid_mag_all = vector_mag_mas(psf_resid_vec_all)
rubin_center_shift_mag_all = vector_mag_mas(rubin_center_shift_vec_all)
vis_center_shift_mag_all = vector_mag_mas(vis_center_shift_vec_all)
centering_delta_mag_all = vector_mag_mas(centering_delta_vec_all)
bulk_mags_by_tile = np.asarray(bulk_mags_by_tile, dtype=np.float32)

# Sanity check: offset change should equal VIS recentering minus Rubin recentering.
identity_error = vector_mag_mas(
    centering_delta_vec_all - (vis_center_shift_vec_all - rubin_center_shift_vec_all)
)

print('=' * 66)
print('CENTERING DIAGNOSTIC')
print('=' * 66)
print(f'Sources pooled: {len(psf_offset_mag_all)} from {len(results)} tiles')
print('')
print('Offset magnitude under different centroid definitions:')
print(f'  Detection centroids:       median {np.median(det_offset_mag_all):6.1f} mas')
print(f'  Flux-weighted centroids:   median {np.median(flux_offset_mag_all):6.1f} mas')
print(f'  PSF-fit centroids:         median {np.median(psf_offset_mag_all):6.1f} mas')
print('')
print('How much recentering changes the measurement:')
print(f'  Rubin detection -> PSF:    median {np.median(rubin_center_shift_mag_all):6.1f} mas')
print(f'  VIS detection -> PSF:      median {np.median(vis_center_shift_mag_all):6.1f} mas')
print(f'  Offset change PSF - det:   median {np.median(centering_delta_mag_all):6.1f} mas')
print(f'  Vector identity check:     median {np.median(identity_error):.3e} mas')
print('')
print('Smooth field versus source-level residual:')
print(f'  Per-tile bulk field:       median {np.median(bulk_mags_by_tile):6.1f} mas')
print(f'  Residual after bulk:       median {np.median(psf_resid_mag_all):6.1f} mas')
print('')
print('Conclusion: the centering term is the same scale as the large offset,')
print('while the smooth concordance-field term is only a few mas in this sample.')
print('=' * 66)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# Panel 1: offset distribution under centroid definitions.
ax = axes[0]
bins = np.linspace(0, 180, 60)
for arr, label, color in [
    (det_offset_mag_all, f'detection med={np.median(det_offset_mag_all):.1f}', '0.45'),
    (flux_offset_mag_all, f'flux med={np.median(flux_offset_mag_all):.1f}', 'seagreen'),
    (psf_offset_mag_all, f'PSF med={np.median(psf_offset_mag_all):.1f}', 'steelblue'),
]:
    ax.hist(arr, bins=bins, histtype='step', lw=1.8, density=True, label=label, color=color)
ax.set_xlabel('|VIS - Rubin offset| [mas]')
ax.set_ylabel('Density')
ax.set_title('Offset depends on centroid definition', fontsize=10)
ax.legend(fontsize=8)
ax.set_xlim(0, 180)

# Panel 2: centering term versus final PSF-fit offset.
ax = axes[1]
valid_center = np.isfinite(psf_offset_mag_all) & np.isfinite(centering_delta_mag_all)
sc = ax.scatter(
    centering_delta_mag_all[valid_center], psf_offset_mag_all[valid_center],
    s=5, alpha=0.25, c=np.log10(np.clip(centering_delta_mag_all[valid_center], 1, None)),
    cmap='viridis', rasterized=True,
)
lim = min(180, np.nanpercentile(np.r_[centering_delta_mag_all, psf_offset_mag_all], 98))
ax.plot([0, lim], [0, lim], 'k--', lw=1, alpha=0.5, label='1:1')
ax.set_xlabel('|offset change from recentering| [mas]')
ax.set_ylabel('|PSF-fit VIS - Rubin offset| [mas]')
ax.set_title('Measured offset moves with centering', fontsize=10)
ax.set_xlim(0, lim); ax.set_ylim(0, lim)
ax.legend(fontsize=8)
plt.colorbar(sc, ax=ax, label='log10 centering term', shrink=0.8)

# Panel 3: compare the terms in the budget.
ax = axes[2]
labels = ['bulk\nfield', 'post-bulk\nresidual', 'Rubin\nrecenter', 'VIS\nrecenter', 'offset\nchange']
values = [
    np.median(bulk_mags_by_tile),
    np.median(psf_resid_mag_all),
    np.median(rubin_center_shift_mag_all),
    np.median(vis_center_shift_mag_all),
    np.median(centering_delta_mag_all),
]
colors = ['0.35', 'steelblue', 'indianred', 'seagreen', 'purple']
bars = ax.bar(labels, values, color=colors, alpha=0.78)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 2, f'{val:.1f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Median magnitude [mas]')
ax.set_title('Centering dwarfs the smooth field', fontsize=10)
ax.set_ylim(0, max(values) * 1.35)
ax.grid(axis='y', alpha=0.2)

plt.tight_layout()
plt.savefig('astrometry_centering_diagnostic.png', dpi=200, bbox_inches='tight')
plt.show()


## 3. Centroiding quality - VIS and Rubin cutouts

For the first 3 tiles, show 8 sources each: VIS and Rubin r-band cutouts side by side, with the detection centroid (red cross), PSF-fit centroid (cyan cross), and flux-weighted centroid (yellow cross). The PSF-fit Gaussian FWHM is shown as a cyan circle.

**What to look for:**
- Are the three centroids consistent? Large disagreement means the object has an ambiguous center.
- Is the PSF-fit circle well-centered on the peak? Off-center means blend, galaxy, or a local maximum that is not the object center.
- How much does the object center move between the multi-band detection image, the Rubin target band, and VIS?


In [ ]:
def _cutout(img, x, y, r=15):
    """Extract a small cutout, clipped to image bounds."""
    H, W = img.shape
    x0, y0 = int(round(x)) - r, int(round(y)) - r
    x1, y1 = x0 + 2*r + 1, y0 + 2*r + 1
    # Clip
    xa, ya = max(0, x0), max(0, y0)
    xb, yb = min(W, x1), min(H, y1)
    cut = np.full((2*r+1, 2*r+1), np.nan, dtype=np.float32)
    cut[ya - y0 : ya - y0 + yb - ya, xa - x0 : xa - x0 + xb - xa] = img[ya:yb, xa:xb]
    return cut, x - x0, y - y0  # cutout and local centroid coords


def _display_norm(img):
    """Arcsinh stretch for display."""
    v = np.nan_to_num(img, nan=0.0)
    med = np.median(v)
    sig = max(1.4826 * np.median(np.abs(v - med)), 1e-8)
    return np.arcsinh((v - med) / sig)


def plot_centroid_cutouts(m, tile, n_show=8, title_prefix=''):
    """Show VIS and Rubin cutouts side by side for n_show matched sources."""
    vis_img = tile['vis_img']
    rubin_target = m['rubin_target']

    # Sort by VIS SNR descending — show best sources first
    order = np.argsort(-m['vis_snr'])[:n_show]

    fig, axes = plt.subplots(n_show, 2, figsize=(6, 2.2 * n_show))
    if n_show == 1:
        axes = axes[None, :]

    for row, idx in enumerate(order):
        # ---- VIS cutout ----
        vx_d, vy_d = m['vis_det_xy'][idx]      # detection
        vx_p, vy_p = m['vis_psf_xy'][idx]      # PSF-fit
        vx_f, vy_f = m['vis_flux_xy'][idx]     # flux-weighted

        cut_v, lx_d, ly_d = _cutout(vis_img, vx_d, vy_d, r=15)
        # Local coords for PSF and flux centroids
        lx_p = vx_p - (int(round(vx_d)) - 15)
        ly_p = vy_p - (int(round(vy_d)) - 15)
        lx_f = vx_f - (int(round(vx_d)) - 15)
        ly_f = vy_f - (int(round(vy_d)) - 15)

        ax = axes[row, 0]
        ax.imshow(_display_norm(cut_v), cmap='gray_r', origin='lower')
        ax.plot(lx_d, ly_d, 'r+', ms=8, mew=1.5, label='detect')
        ax.plot(lx_p, ly_p, 'c+', ms=8, mew=1.5, label='PSF-fit')
        ax.plot(lx_f, ly_f, '+', color='gold', ms=8, mew=1.5, label='flux-wt')
        # PSF FWHM circle from centroid sigma
        if np.isfinite(m['vis_csig_px'][idx]) and m['vis_csig_px'][idx] < 20:
            fwhm = m['vis_csig_px'][idx] * m['vis_snr'][idx] * 2.3548 / max(m['vis_snr'][idx], 1)
            # Actually: csig = sigma_psf / SNR, so sigma_psf = csig * SNR
            # FWHM = sigma_psf * 2.3548
            sig_psf = m['vis_csig_px'][idx] * max(m['vis_snr'][idx], 1)
            fwhm = sig_psf * 2.3548
            if 1 < fwhm < 20:
                circ = Circle((lx_p, ly_p), fwhm / 2, fill=False, ec='cyan', lw=0.8, ls='--')
                ax.add_patch(circ)
        snr_v = m['vis_snr'][idx]
        ax.set_title(f'VIS  SNR={snr_v:.0f}  |off|={m["offset_mag"][idx]:.0f}mas', fontsize=8)
        ax.set_xlim(5, 26); ax.set_ylim(5, 26)
        ax.tick_params(labelsize=6)
        if row == 0:
            ax.legend(fontsize=5, loc='upper right')

        # ---- Rubin cutout ----
        rx_d, ry_d = m['rubin_det_xy'][idx]
        rx_p, ry_p = m['rubin_psf_xy'][idx]
        rx_f, ry_f = m['rubin_flux_xy'][idx]

        cut_r, lrx_d, lry_d = _cutout(rubin_target, rx_d, ry_d, r=15)
        lrx_p = rx_p - (int(round(rx_d)) - 15)
        lry_p = ry_p - (int(round(ry_d)) - 15)
        lrx_f = rx_f - (int(round(rx_d)) - 15)
        lry_f = ry_f - (int(round(ry_d)) - 15)

        ax = axes[row, 1]
        ax.imshow(_display_norm(cut_r), cmap='gray_r', origin='lower')
        ax.plot(lrx_d, lry_d, 'r+', ms=8, mew=1.5)
        ax.plot(lrx_p, lry_p, 'c+', ms=8, mew=1.5)
        ax.plot(lrx_f, lry_f, '+', color='gold', ms=8, mew=1.5)
        if np.isfinite(m['rubin_csig_px'][idx]) and m['rubin_csig_px'][idx] < 20:
            sig_psf_r = m['rubin_csig_px'][idx] * max(m['rubin_snr'][idx], 1)
            fwhm_r = sig_psf_r * 2.3548
            if 1 < fwhm_r < 20:
                circ = Circle((lrx_p, lry_p), fwhm_r / 2, fill=False, ec='cyan', lw=0.8, ls='--')
                ax.add_patch(circ)

        # VIS PSF-fit centroid projected into the Rubin pixel frame via WCS,
        # then into Rubin-cutout local coordinates. Arrow on the Rubin cutout
        # goes from the Rubin PSF centre to where VIS says the centre is.
        rwcs = tile['rwcs']; vwcs = tile['vwcs']
        v_ra_p, v_dec_p = vwcs.wcs_pix2world(vx_p, vy_p, 0)
        rx_from_vis, ry_from_vis = rwcs.wcs_world2pix(v_ra_p, v_dec_p, 0)
        lrx_vproj = float(rx_from_vis) - (int(round(rx_d)) - 15)
        lry_vproj = float(ry_from_vis) - (int(round(ry_d)) - 15)
        off_mas = float(m['offset_mag'][idx])
        # Only mark if the projected point sits inside the cutout.
        if 0 <= lrx_vproj <= 30 and 0 <= lry_vproj <= 30:
            ax.plot(lrx_vproj, lry_vproj, 'o', color='lime',
                    ms=5, mew=0, label='VIS (proj)' if row == 0 else None)
            ax.text(0.97, 0.03, f'{off_mas:.0f} mas',
                    transform=ax.transAxes, ha='right', va='bottom',
                    fontsize=6.5, color='lime',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='black',
                              alpha=0.55, edgecolor='none'))

        snr_r = m['rubin_snr'][idx]
        ax.set_title(f'Rubin {m["target_band"]}  SNR={snr_r:.0f}', fontsize=8)
        ax.set_xlim(5, 26); ax.set_ylim(5, 26)
        ax.tick_params(labelsize=6)
        if row == 0:
            ax.legend(fontsize=5, loc='upper right')

    fig.suptitle(f'{title_prefix}Centroid comparison (sorted by VIS SNR)', fontsize=10, y=1.01)
    plt.tight_layout()
    plt.show()


# Show for first 3 tiles
for tid in list(results.keys())[:3]:
    m = results[tid]
    plot_centroid_cutouts(m, m['tile'], n_show=8, title_prefix=f'{tid}\n')

## 4. Offset vector field per tile

Show the matched source positions on the VIS image with arrows indicating the (VIS - Rubin) offset direction. Arrows are colored by offset magnitude. A smooth concordance field should show arrows pointing in a coherent direction that varies slowly. Random-looking arrows indicate object-level centering scatter dominating over the field signal.


In [ ]:
def plot_offset_vector_field(m, tile, title=''):
    """Quiver plot of offsets overlaid on VIS image."""
    vis_img = tile['vis_img']
    H, W = vis_img.shape

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # --- Panel 1: VIS image with offset quivers ---
    ax = axes[0]
    disp = _display_norm(vis_img)
    ax.imshow(disp, cmap='gray_r', origin='lower', vmin=-2, vmax=8)
    # Scale arrows: offsets are in arcsec, VIS pixel = 0.1", so multiply by 10 for pixel units
    # Then scale up for visibility
    scale_factor = 10.0 / 0.1  # arcsec → VIS pixels, then amplify
    vx = m['vis_psf_xy'][:, 0]
    vy = m['vis_psf_xy'][:, 1]
    dx = m['dra'] * scale_factor
    dy = m['ddec'] * scale_factor
    mag = m['offset_mag']

    q = ax.quiver(vx, vy, dx, dy, mag, cmap='coolwarm', scale=1, scale_units='xy',
                  angles='xy', clim=(0, 100), width=0.002, headwidth=4)
    plt.colorbar(q, ax=ax, label='|offset| [mas]', shrink=0.8)
    ax.set_title(f'Offset vectors on VIS\n(amplified {scale_factor:.0f}×)', fontsize=9)
    ax.set_xlim(0, W); ax.set_ylim(0, H)
    ax.set_aspect('equal')

    # --- Panel 2: ΔRA vs ΔDec scatter, colored by SNR ---
    ax = axes[1]
    snr_comb = np.sqrt(m['rubin_snr'] * m['vis_snr'])  # geometric mean SNR
    sc = ax.scatter(m['dra'] * 1000, m['ddec'] * 1000, c=np.log10(np.clip(snr_comb, 1, None)),
                    s=10, cmap='viridis', alpha=0.7, vmin=0.5, vmax=3)
    plt.colorbar(sc, ax=ax, label='log₁₀(√(SNR_R × SNR_V))', shrink=0.8)
    # Median offset cross
    med_dra = np.median(m['dra']) * 1000
    med_ddec = np.median(m['ddec']) * 1000
    ax.axhline(med_ddec, color='red', lw=0.5, alpha=0.5)
    ax.axvline(med_dra, color='red', lw=0.5, alpha=0.5)
    ax.plot(med_dra, med_ddec, 'rx', ms=10, mew=2)
    ax.set_xlabel('ΔRA* [mas]'); ax.set_ylabel('ΔDec [mas]')
    ax.set_title(f'Offset scatter (median: {med_dra:.1f}, {med_ddec:.1f} mas)\n'
                 f'{m["n_matched"]} sources, colored by combined SNR', fontsize=9)
    ax.set_aspect('equal')
    lim = max(np.percentile(np.abs(m['dra'] * 1000), 95),
              np.percentile(np.abs(m['ddec'] * 1000), 95)) * 1.3
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.grid(alpha=0.2)

    # --- Panel 3: PSF-fit vs flux-weighted offset comparison ---
    ax = axes[2]
    ax.scatter(m['offset_mag'], m['offset_mag_flux'], s=8, alpha=0.5, c='steelblue')
    lim2 = max(np.percentile(m['offset_mag'], 95), np.percentile(m['offset_mag_flux'], 95)) * 1.2
    ax.plot([0, lim2], [0, lim2], 'k--', lw=0.8, alpha=0.5)
    ax.set_xlabel('|offset| PSF-fit [mas]'); ax.set_ylabel('|offset| flux-weighted [mas]')
    ax.set_title('PSF-fit vs flux-weighted centroiding', fontsize=9)
    ax.set_xlim(0, lim2); ax.set_ylim(0, lim2)
    ax.set_aspect('equal')
    ax.grid(alpha=0.2)
    # Medians
    med_psf = np.median(m['offset_mag'])
    med_flux = np.median(m['offset_mag_flux'])
    ax.text(0.05, 0.92, f'PSF-fit median: {med_psf:.1f} mas\nFlux-wt median: {med_flux:.1f} mas',
            transform=ax.transAxes, fontsize=8, va='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    fig.suptitle(title, fontsize=11, y=1.02)
    plt.tight_layout()
    plt.show()


for tid in list(results.keys())[:4]:
    plot_offset_vector_field(results[tid], results[tid]['tile'], title=tid)

## 5. Offset vs SNR - centroid noise check

King (1983) / SITCOMTN-159 predict: centroid noise sigma is proportional to FWHM / SNR. After subtracting the bulk systematic offset per tile, the **residual scatter** should follow this scaling.

We pool all sources from all sample tiles, subtract per-tile median offsets (removing the bulk concordance), and plot residual scatter vs SNR. If the scatter tracks the theoretical curve, centroid noise / centering precision is the bottleneck. If it plateaus at high SNR, there is an additional systematic floor.


In [ ]:
# Pool all sources, subtract per-tile bulk offset to isolate centroid scatter
all_snr_rubin = []
all_snr_vis = []
all_resid_mag = []    # residual |offset| after removing tile median (mas)
all_raw_mag = []      # raw |offset| (mas)
all_fwhm_vis = []     # PSF FWHM from VIS fit (pixels)
all_csig_rubin = []   # centroid sigma from Rubin PSF fit (pixels)
all_csig_vis = []     # centroid sigma from VIS PSF fit (pixels)

for tid, m in results.items():
    # Per-tile bulk systematic
    med_dra = np.median(m['dra'])
    med_ddec = np.median(m['ddec'])
    # Residual after removing bulk offset
    resid_dra = (m['dra'] - med_dra) * 1000   # mas
    resid_ddec = (m['ddec'] - med_ddec) * 1000
    resid_mag = np.hypot(resid_dra, resid_ddec)

    all_snr_rubin.append(m['rubin_snr'])
    all_snr_vis.append(m['vis_snr'])
    all_resid_mag.append(resid_mag)
    all_raw_mag.append(m['offset_mag'])
    all_csig_rubin.append(m['rubin_csig_px'])
    all_csig_vis.append(m['vis_csig_px'])
    # Recover PSF FWHM: csig = sigma_psf / SNR → sigma_psf = csig * SNR → FWHM = 2.35 * sigma_psf
    fwhm_v = m['vis_csig_px'] * np.maximum(m['vis_snr'], 1) * 2.3548
    all_fwhm_vis.append(fwhm_v)

all_snr_rubin = np.concatenate(all_snr_rubin)
all_snr_vis = np.concatenate(all_snr_vis)
all_resid_mag = np.concatenate(all_resid_mag)
all_raw_mag = np.concatenate(all_raw_mag)
all_fwhm_vis = np.concatenate(all_fwhm_vis)
all_csig_rubin = np.concatenate(all_csig_rubin)
all_csig_vis = np.concatenate(all_csig_vis)

# Combined SNR: geometric mean
all_snr_comb = np.sqrt(np.maximum(all_snr_rubin, 1) * np.maximum(all_snr_vis, 1))

print(f'Pooled {len(all_snr_comb)} sources from {len(results)} tiles')
print(f'  SNR range: Rubin [{np.percentile(all_snr_rubin,5):.0f}, {np.percentile(all_snr_rubin,95):.0f}], '
      f'VIS [{np.percentile(all_snr_vis,5):.0f}, {np.percentile(all_snr_vis,95):.0f}]')
print(f'  Raw |offset| median: {np.median(all_raw_mag):.1f} mas')
print(f'  Residual |offset| median (after tile median subtracted): {np.median(all_resid_mag):.1f} mas')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# --- Panel 1: Residual scatter vs combined SNR ---
ax = axes[0]
valid = np.isfinite(all_snr_comb) & (all_snr_comb > 1)
snr_v = all_snr_comb[valid]
res_v = all_resid_mag[valid]

ax.scatter(snr_v, res_v, s=3, alpha=0.15, c='steelblue', rasterized=True)

# Binned running median + 68th percentile
snr_bins = np.logspace(np.log10(2), np.log10(snr_v.max()), 20)
bin_centers, bin_medians, bin_p68 = [], [], []
for i in range(len(snr_bins) - 1):
    mask = (snr_v >= snr_bins[i]) & (snr_v < snr_bins[i+1])
    if mask.sum() < 5:
        continue
    bin_centers.append(np.sqrt(snr_bins[i] * snr_bins[i+1]))
    bin_medians.append(np.median(res_v[mask]))
    bin_p68.append(np.percentile(res_v[mask], 68))

ax.plot(bin_centers, bin_medians, 'r-o', ms=4, lw=1.5, label='median')
ax.plot(bin_centers, bin_p68, 'r--s', ms=3, lw=1, label='p68')

# Theoretical: σ ∝ FWHM / SNR (combined for both instruments)
# Rubin FWHM ~ 3 VIS px → 0.3" = 300 mas; VIS FWHM ~ 4 VIS px → 0.4" = 400 mas
# Combined centroid noise: σ = sqrt(σ_R² + σ_V²) ~ sqrt((300/SNR)² + (400/SNR)²) / sqrt(2)
# Approximate: ~ 350 / SNR (mas)
snr_theory = np.logspace(0.3, 3, 100)
sigma_theory = np.sqrt((300 / snr_theory)**2 + (200 / snr_theory)**2)
ax.plot(snr_theory, sigma_theory, 'k--', lw=1, alpha=0.7,
        label='Theory: √((300/SNR)² + (200/SNR)²)')
# With 5 mas floor
sigma_floor = np.sqrt(sigma_theory**2 + 5**2)
ax.plot(snr_theory, sigma_floor, 'k:', lw=1, alpha=0.5, label='+ 5 mas floor')

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Combined SNR = √(SNR_R × SNR_V)', fontsize=10)
ax.set_ylabel('Residual |offset| after tile-median [mas]', fontsize=10)
ax.set_title('Centroid noise vs SNR\n(bulk systematic per tile removed)', fontsize=10)
ax.legend(fontsize=7, loc='upper right')
ax.set_xlim(2, 3000); ax.set_ylim(1, 300)
ax.grid(alpha=0.2, which='both')

# --- Panel 2: Separate Rubin vs VIS SNR ---
ax = axes[1]
ax.scatter(all_snr_rubin[valid], res_v, s=3, alpha=0.12, c='indianred', label='Rubin SNR', rasterized=True)
ax.scatter(all_snr_vis[valid], res_v, s=3, alpha=0.12, c='steelblue', label='VIS SNR', rasterized=True)
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('SNR (per instrument)', fontsize=10)
ax.set_ylabel('Residual |offset| [mas]', fontsize=10)
ax.set_title('Rubin SNR vs VIS SNR separately', fontsize=10)
ax.legend(fontsize=8, markerscale=3)
ax.set_xlim(1, 3000); ax.set_ylim(1, 300)
ax.grid(alpha=0.2, which='both')

# --- Panel 3: FWHM distribution (star vs galaxy proxy) ---
ax = axes[2]
fwhm_valid = all_fwhm_vis[valid & np.isfinite(all_fwhm_vis) & (all_fwhm_vis > 0) & (all_fwhm_vis < 30)]
ax.hist(fwhm_valid, bins=np.linspace(0, 20, 60), color='steelblue', alpha=0.7, edgecolor='white', lw=0.3)
ax.axvline(np.median(fwhm_valid), color='red', lw=1.5, ls='--', label=f'median = {np.median(fwhm_valid):.1f} px')
# Mark typical PSF FWHM for point sources
ax.axvline(4.0, color='green', lw=1, ls=':', label='Expected PSF FWHM ~4 px')
ax.set_xlabel('VIS PSF-fit FWHM [pixels]', fontsize=10)
ax.set_ylabel('Count', fontsize=10)
ax.set_title('Source size distribution\n(FWHM > PSF = resolved / galaxy)', fontsize=10)
ax.legend(fontsize=8)
ax.set_xlim(0, 20)

plt.tight_layout()
plt.savefig('astrometry_snr_diagnostic.png', dpi=200, bbox_inches='tight')
plt.show()

## 6. Stars vs galaxies - morphology dependence

Split sources by FWHM: compact (FWHM < 1.5 x PSF) are likely stars, extended sources are likely galaxies. If galaxies dominate the large-offset tail, morphology is part of the centering problem. If stars have the same scale, the problem is broader centroid scatter rather than only galaxy color gradients.


In [ ]:
# Star/galaxy split: use VIS FWHM relative to the expected PSF FWHM
PSF_FWHM_VIS = 4.0  # typical VIS PSF FWHM in pixels (~0.4")
STAR_THRESHOLD = PSF_FWHM_VIS * 1.5  # sources below this are "point-like"

fwhm_all = all_fwhm_vis.copy()
fwhm_all[~np.isfinite(fwhm_all)] = 999  # mark failed fits as extended

is_star = (fwhm_all > 0) & (fwhm_all < STAR_THRESHOLD) & valid
is_galaxy = (fwhm_all >= STAR_THRESHOLD) & (fwhm_all < 30) & valid

print(f'Stars (FWHM < {STAR_THRESHOLD:.1f} px): {is_star.sum()}')
print(f'Galaxies / extended (FWHM >= {STAR_THRESHOLD:.1f} px): {is_galaxy.sum()}')
print(f'PSF-fit failed (FWHM=inf): {(~np.isfinite(all_fwhm_vis) & valid).sum()}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

# --- Panel 1: Residual scatter: stars vs galaxies ---
ax = axes[0]
ax.scatter(all_snr_comb[is_galaxy], all_resid_mag[is_galaxy], s=4, alpha=0.15,
           c='indianred', label=f'Extended ({is_galaxy.sum()})', rasterized=True)
ax.scatter(all_snr_comb[is_star], all_resid_mag[is_star], s=4, alpha=0.2,
           c='steelblue', label=f'Point-like ({is_star.sum()})', rasterized=True)

# Binned medians for each class
for mask, color, ls in [(is_star, 'steelblue', '-'), (is_galaxy, 'indianred', '--')]:
    s, r = all_snr_comb[mask], all_resid_mag[mask]
    bins = np.logspace(np.log10(max(2, s.min())), np.log10(s.max()), 15)
    cx, cy = [], []
    for i in range(len(bins)-1):
        m2 = (s >= bins[i]) & (s < bins[i+1])
        if m2.sum() >= 5:
            cx.append(np.sqrt(bins[i]*bins[i+1]))
            cy.append(np.median(r[m2]))
    ax.plot(cx, cy, color=color, ls=ls, lw=2, marker='o', ms=4)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Combined SNR'); ax.set_ylabel('Residual |offset| [mas]')
ax.set_title('Stars vs galaxies: residual scatter', fontsize=10)
ax.legend(fontsize=8, markerscale=2)
ax.set_xlim(2, 3000); ax.set_ylim(1, 300)
ax.grid(alpha=0.2, which='both')

# --- Panel 2: Histogram comparison (stars vs galaxies) ---
ax = axes[1]
bins = np.linspace(0, 150, 50)
ax.hist(all_resid_mag[is_star], bins=bins, alpha=0.5, color='steelblue', density=True,
        label=f'Stars (med={np.median(all_resid_mag[is_star]):.1f} mas)')
ax.hist(all_resid_mag[is_galaxy], bins=bins, alpha=0.5, color='indianred', density=True,
        label=f'Galaxies (med={np.median(all_resid_mag[is_galaxy]):.1f} mas)')
ax.set_xlabel('Residual |offset| [mas]'); ax.set_ylabel('Density')
ax.set_title('Residual distribution by morphology', fontsize=10)
ax.legend(fontsize=8)
ax.set_xlim(0, 150)

# --- Panel 3: Raw offset (not residual) by morphology ---
ax = axes[2]
ax.hist(all_raw_mag[is_star], bins=np.linspace(0, 200, 50), alpha=0.5, color='steelblue', density=True,
        label=f'Stars (med={np.median(all_raw_mag[is_star]):.1f} mas)')
ax.hist(all_raw_mag[is_galaxy], bins=np.linspace(0, 200, 50), alpha=0.5, color='indianred', density=True,
        label=f'Galaxies (med={np.median(all_raw_mag[is_galaxy]):.1f} mas)')
ax.set_xlabel('Raw |offset| [mas]'); ax.set_ylabel('Density')
ax.set_title('Raw offset distribution by morphology\n(includes concordance field)', fontsize=10)
ax.legend(fontsize=8)
ax.set_xlim(0, 200)

plt.tight_layout()
plt.savefig('astrometry_morphology_diagnostic.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Residuals after smooth polynomial field removal

Fit a low-order 2D polynomial to per-tile offsets and subtract it. What remains is either:
- Centroid noise / centering scatter (random, source-by-source)
- Higher-order field structure (correlated, spatial pattern)
- Match contamination (extreme outliers)

If this residual is still tens of mas while the fitted field is only a few mas, a concordance file is not the limiting correction.


In [ ]:
def fit_poly_field(x, y, dra, ddec, order=2):
    """Fit a 2D polynomial to the offset field and return residuals.
    
    Uses normalized coordinates and weighted least squares (weight by 1/offset_mag
    to downweight outliers).
    """
    # Normalize coordinates to [-1, 1]
    xn = (x - x.mean()) / max(x.std(), 1e-8)
    yn = (y - y.mean()) / max(y.std(), 1e-8)
    
    # Build polynomial design matrix (up to given order)
    cols = []
    for p in range(order + 1):
        for q in range(order + 1 - p):
            cols.append(xn**p * yn**q)
    A = np.column_stack(cols)
    
    # Solve for dRA and dDec independently
    dra_coeffs, _, _, _ = np.linalg.lstsq(A, dra, rcond=None)
    ddec_coeffs, _, _, _ = np.linalg.lstsq(A, ddec, rcond=None)
    
    dra_fit = A @ dra_coeffs
    ddec_fit = A @ ddec_coeffs
    
    return dra - dra_fit, ddec - ddec_fit, dra_fit, ddec_fit


# Process each tile: fit polynomial, compute residuals
poly_results = {}
for tid, m in results.items():
    vx = m['vis_psf_xy'][:, 0]
    vy = m['vis_psf_xy'][:, 1]
    
    resid_dra, resid_ddec, fit_dra, fit_ddec = fit_poly_field(
        vx, vy, m['dra'], m['ddec'], order=2)
    
    poly_results[tid] = {
        'resid_dra': resid_dra * 1000,    # mas
        'resid_ddec': resid_ddec * 1000,
        'fit_dra': fit_dra * 1000,
        'fit_ddec': fit_ddec * 1000,
        'resid_mag': np.hypot(resid_dra, resid_ddec) * 1000,
    }

# Show a few tiles: original offsets vs polynomial fit vs residuals
fig, axes = plt.subplots(min(4, len(results)), 3, figsize=(16, 4 * min(4, len(results))))
if len(results) == 1:
    axes = axes[None, :]

for row, tid in enumerate(list(results.keys())[:4]):
    m = results[tid]
    p = poly_results[tid]
    vx = m['vis_psf_xy'][:, 0]
    vy = m['vis_psf_xy'][:, 1]
    
    # Panel 1: Raw offsets as quiver
    ax = axes[row, 0]
    q = ax.quiver(vx, vy, m['dra'] * 1000, m['ddec'] * 1000,
                  m['offset_mag'], cmap='coolwarm', clim=(0, 100),
                  scale=500, width=0.004)
    ax.set_title(f'{tid}\nRaw offsets (med={np.median(m["offset_mag"]):.0f} mas)', fontsize=8)
    ax.set_aspect('equal')
    
    # Panel 2: Polynomial fit (the smooth field)
    ax = axes[row, 1]
    fit_mag = np.hypot(p['fit_dra'], p['fit_ddec'])
    q = ax.quiver(vx, vy, p['fit_dra'], p['fit_ddec'],
                  fit_mag, cmap='coolwarm', clim=(0, 100),
                  scale=500, width=0.004)
    ax.set_title(f'Polynomial fit (order 2)\nmed={np.median(fit_mag):.0f} mas', fontsize=8)
    ax.set_aspect('equal')
    
    # Panel 3: Residuals after polynomial removal
    ax = axes[row, 2]
    q = ax.quiver(vx, vy, p['resid_dra'], p['resid_ddec'],
                  p['resid_mag'], cmap='coolwarm', clim=(0, 80),
                  scale=500, width=0.004)
    ax.set_title(f'Residuals\nmed={np.median(p["resid_mag"]):.0f} mas, '
                 f'p68={np.percentile(p["resid_mag"], 68):.0f} mas', fontsize=8)
    ax.set_aspect('equal')

plt.suptitle('Raw offsets → Polynomial field fit → Residuals (per tile)', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 8. Multi-band comparison - band-centering offsets

For sources detected in both Rubin r and Rubin i bands, compare the measured offsets to the same VIS anchor. This is not purely a galaxy chromaticity test: if point-like sources show the same r-vs-i scale, then the measurement is sensitive to per-band centroid realization and centering noise, not only galaxy color gradients.


In [ ]:
# Run matching with Rubin i-band for comparison (reuse same tiles)
results_i = {}
for tid in list(results.keys())[:8]:  # first 8 tiles to keep it fast
    rp, ep = pair_map[tid]
    try:
        tile = load_tile(rp, ep)
        m_i = run_matching_pipeline(tile, target_band='i')
        if m_i is not None and m_i['n_matched'] >= 10:
            results_i[tid] = m_i
    except Exception:
        pass

# For tiles with both r and i results, compare offsets at matched positions
# We match by closest VIS position (same source detected in both bands)
chromatic_diffs = []
for tid in results_i:
    if tid not in results:
        continue
    mr = results[tid]
    mi = results_i[tid]
    
    # Match by VIS pixel distance (same VIS source, different Rubin band)
    from scipy.spatial import cKDTree
    tree_r = cKDTree(mr['vis_psf_xy'])
    tree_i = cKDTree(mi['vis_psf_xy'])
    
    dists, idx_r = tree_i.query(mi['vis_psf_xy'])  # for each i-source, find closest r-source
    # Wait, that queries i against i. Let me fix:
    dists, idx_r = tree_r.query(mi['vis_psf_xy'])
    
    close = dists < 3.0  # within 3 VIS pixels = same source
    if close.sum() < 5:
        continue
    
    # Chromatic offset = (r-band offset) - (i-band offset)
    dra_diff = mr['dra'][idx_r[close]] - mi['dra'][close]
    ddec_diff = mr['ddec'][idx_r[close]] - mi['ddec'][close]
    chrom_mag = np.hypot(dra_diff, ddec_diff) * 1000  # mas
    
    # Also get the VIS FWHM from the r result (star/galaxy proxy)
    fwhm_v = all_fwhm_vis_local = mr['vis_csig_px'][idx_r[close]] * np.maximum(mr['vis_snr'][idx_r[close]], 1) * 2.3548
    snr_v = mr['vis_snr'][idx_r[close]]
    
    for j in range(len(dra_diff)):
        chromatic_diffs.append({
            'dra_diff': dra_diff[j] * 1000,
            'ddec_diff': ddec_diff[j] * 1000,
            'chrom_mag': chrom_mag[j],
            'fwhm_vis': fwhm_v[j],
            'snr_vis': snr_v[j],
        })

if chromatic_diffs:
    cd = {k: np.array([d[k] for d in chromatic_diffs]) for k in chromatic_diffs[0]}
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Panel 1: r-i chromatic offset histogram
    ax = axes[0]
    ax.hist(cd['chrom_mag'], bins=np.linspace(0, 80, 40), color='mediumpurple', alpha=0.7,
            edgecolor='white', lw=0.3)
    med = np.median(cd['chrom_mag'])
    ax.axvline(med, color='red', ls='--', lw=1.5, label=f'median = {med:.1f} mas')
    ax.set_xlabel('|Δoffset(r) - Δoffset(i)| [mas]')
    ax.set_ylabel('Count')
    ax.set_title(f'Chromatic offset: Rubin r vs i\n{len(cd["chrom_mag"])} matched sources', fontsize=10)
    ax.legend(fontsize=8)
    
    # Panel 2: Chromatic offset vs FWHM (star vs galaxy)
    ax = axes[1]
    fwhm_ok = np.isfinite(cd['fwhm_vis']) & (cd['fwhm_vis'] > 0) & (cd['fwhm_vis'] < 25)
    ax.scatter(cd['fwhm_vis'][fwhm_ok], cd['chrom_mag'][fwhm_ok],
              s=5, alpha=0.4, c='mediumpurple', rasterized=True)
    ax.axvline(STAR_THRESHOLD, color='green', ls=':', lw=1, label=f'Star/galaxy cut = {STAR_THRESHOLD:.1f} px')
    ax.set_xlabel('VIS FWHM [px]'); ax.set_ylabel('Chromatic |Δoffset| [mas]')
    ax.set_title('Chromatic offset vs source size', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlim(0, 20); ax.set_ylim(0, min(100, np.percentile(cd['chrom_mag'], 98)))
    
    # Panel 3: Chromatic offset vs SNR
    ax = axes[2]
    ax.scatter(cd['snr_vis'], cd['chrom_mag'], s=5, alpha=0.4, c='mediumpurple', rasterized=True)
    ax.set_xscale('log')
    ax.set_xlabel('VIS SNR'); ax.set_ylabel('Chromatic |Δoffset| [mas]')
    ax.set_title('Chromatic offset vs SNR', fontsize=10)
    ax.set_ylim(0, min(100, np.percentile(cd['chrom_mag'], 98)))
    ax.grid(alpha=0.2)
    
    plt.tight_layout()
    plt.savefig('astrometry_chromatic_diagnostic.png', dpi=200, bbox_inches='tight')
    plt.show()
    
    # Summary stats
    stars = cd['fwhm_vis'] < STAR_THRESHOLD
    gals = (cd['fwhm_vis'] >= STAR_THRESHOLD) & (cd['fwhm_vis'] < 25)
    print(f'\nChromatic offset (r vs i):')
    print(f'  All sources:  median = {np.median(cd["chrom_mag"]):.1f} mas  (N={len(cd["chrom_mag"])})')
    if stars.sum() > 0:
        print(f'  Stars only:   median = {np.median(cd["chrom_mag"][stars]):.1f} mas  (N={stars.sum()})')
    if gals.sum() > 0:
        print(f'  Galaxies:     median = {np.median(cd["chrom_mag"][gals]):.1f} mas  (N={gals.sum()})')
else:
    print('No chromatic comparison possible (need matching sources in both r and i bands)')

## 9. Summary: where does the error budget go?

Break down the total offset into components:
- **Bulk systematic**: per-tile median offset, the smooth concordance-field term
- **Centering definition**: how much the measured offset changes when the source center is redefined
- **Centroid noise**: predicted from the measured SNR and FWHM
- **Residual scatter**: what remains after subtracting the tile-level bulk field


In [ ]:
# Error budget analysis
VIS_PIXEL_SCALE = 0.1   # arcsec/px
RUBIN_PIXEL_SCALE = 0.2  # arcsec/px

# Per-source predicted centroid noise (King formula)
# sigma_centroid_per_axis = FWHM / (2.35 * SNR)
# For the combined offset: sigma_total = sqrt(sigma_rubin^2 + sigma_vis^2)
# In arcsec (on sky):
sigma_rubin_arcsec = all_csig_rubin * RUBIN_PIXEL_SCALE  # csig is in Rubin pixels
sigma_vis_arcsec = all_csig_vis * VIS_PIXEL_SCALE        # csig is in VIS pixels
sigma_combined = np.sqrt(sigma_rubin_arcsec**2 + sigma_vis_arcsec**2) * 1000  # mas

# For sources where PSF fit succeeded
good = valid & np.isfinite(sigma_combined) & (sigma_combined > 0) & (sigma_combined < 500)

# Collect per-tile bulk systematics if the centering cell was not run.
if 'bulk_mags_by_tile' not in globals():
    bulk_mags_by_tile = []
    for tid, m in results.items():
        bulk_mags_by_tile.append(np.hypot(np.median(m['dra']), np.median(m['ddec'])) * 1000)
    bulk_mags_by_tile = np.asarray(bulk_mags_by_tile, dtype=np.float32)
bulk_mag = np.median(bulk_mags_by_tile)

print('=' * 66)
print('ERROR BUDGET SUMMARY')
print('=' * 66)
print(f'Number of sources analyzed: {good.sum()} (from {len(results)} tiles)')
print('')
print('1. SMOOTH BULK SYSTEMATIC (concordance field):')
print(f'   Median per-tile |offset|:      {bulk_mag:.1f} mas')
print('   This is the field-level term a concordance surface can model.')
print('')

if 'centering_delta_mag_all' in globals():
    print('2. CENTERING / CENTROID-DEFINITION TERM:')
    print(f'   Rubin detection -> PSF median: {np.median(rubin_center_shift_mag_all):.1f} mas')
    print(f'   VIS detection -> PSF median:   {np.median(vis_center_shift_mag_all):.1f} mas')
    print(f'   Offset change PSF - det:       {np.median(centering_delta_mag_all):.1f} mas')
    print('   This is source-by-source; a smooth concordance file cannot remove it.')
    print('')

print('3. AFTER BULK REMOVAL (source-level residual scatter):')
resid_good = all_resid_mag[good]
print(f'   Median residual |offset|:      {np.median(resid_good):.1f} mas')
print(f'   p68 residual:                  {np.percentile(resid_good, 68):.1f} mas')
print(f'   p90 residual:                  {np.percentile(resid_good, 90):.1f} mas')
print('')
print('4. PREDICTED CENTROID NOISE (King formula):')
print(f'   Median sigma_centroid:         {np.median(sigma_combined[good]):.1f} mas')
print(f'   p68 sigma_centroid:            {np.percentile(sigma_combined[good], 68):.1f} mas')
print('')

ratio = all_resid_mag[good] / np.clip(sigma_combined[good], 1, None)
print('5. RATIO (residual / predicted centroid noise):')
print(f'   Median ratio:                  {np.median(ratio):.2f}')
print('   ratio near 1 means centroiding/centering is the floor')
print('')

# Stars-only analysis
star_good = good & is_star
if star_good.sum() > 10:
    star_ratio = all_resid_mag[star_good] / np.clip(sigma_combined[star_good], 1, None)
    print(f'6. STARS ONLY ({star_good.sum()} sources):')
    print(f'   Residual median:               {np.median(all_resid_mag[star_good]):.1f} mas')
    print(f'   Predicted noise median:        {np.median(sigma_combined[star_good]):.1f} mas')
    print(f'   Ratio:                         {np.median(star_ratio):.2f}')

# Galaxies / extended analysis
gal_good = good & is_galaxy
if gal_good.sum() > 10:
    gal_ratio = all_resid_mag[gal_good] / np.clip(sigma_combined[gal_good], 1, None)
    print(f'\n7. GALAXIES / EXTENDED ({gal_good.sum()} sources):')
    print(f'   Residual median:               {np.median(all_resid_mag[gal_good]):.1f} mas')
    print(f'   Predicted noise median:        {np.median(sigma_combined[gal_good]):.1f} mas')
    print(f'   Ratio:                         {np.median(gal_ratio):.2f}')

# High-SNR only (best sources, where we should be closest to the floor)
high_snr = good & (all_snr_comb > 100)
if high_snr.sum() > 10:
    print(f'\n8. HIGH SNR ONLY (combined SNR > 100, {high_snr.sum()} sources):')
    print(f'   Residual median:               {np.median(all_resid_mag[high_snr]):.1f} mas')
    print(f'   Predicted noise median:        {np.median(sigma_combined[high_snr]):.1f} mas')
    print('   This is the achievable floor with current centroiding.')

print('')
print('VERDICT: the large offset is dominated by object centering / centroid')
print('definition. The concordance field is useful for global WCS QA, but the')
print('per-object correction has to come from the astrometry head or a better')
print('source-centering/quality model.')
print('=' * 66)


## Interpretation

The answer from this sample is **centering**. The smooth per-tile field is only a few mas, while the source-level residual and the offset change caused by recentering are both tens of mas. The r-vs-i comparison has the same scale for point-like sources, so this is not just galaxy morphology; it is the fact that each band/instrument gives a slightly different object center.

That makes the concordance file useful as a global WCS/concordance QA product, but not as the main correction for these anchors. The astrometry head is useful precisely because it can learn an object-level correction conditioned on the image content and source quality.


## 10. Worst offenders - what goes wrong?

Show cutouts for the 10 sources with the largest residual offsets after bulk removal. These are usually cases where the object center is ambiguous:
- Bad matches or duplicate peaks on the same object
- Blended sources with centroid pulled by a neighbor
- Extended galaxies with wavelength-dependent structure
- Edge effects or artifacts

Understanding these outliers tells us what the matcher and field solver are fighting against.


In [ ]:
# Collect all sources with their tile reference for cutout plotting
all_sources = []
for tid, m in results.items():
    p = poly_results[tid]
    for i in range(m['n_matched']):
        all_sources.append({
            'tid': tid,
            'idx': i,
            'resid_mag': p['resid_mag'][i],
            'raw_mag': m['offset_mag'][i],
            'vis_snr': m['vis_snr'][i],
            'rubin_snr': m['rubin_snr'][i],
            'fwhm_vis': all_fwhm_vis[sum(results[t]['n_matched'] for t in list(results.keys())[:list(results.keys()).index(tid)]) + i] if tid in results else 0,
        })

# Sort by residual magnitude
all_sources.sort(key=lambda s: -s['resid_mag'])

# Show the worst 10
n_worst = 10
fig, axes = plt.subplots(n_worst, 2, figsize=(6, 2.2 * n_worst))

for row in range(n_worst):
    s = all_sources[row]
    tid = s['tid']
    idx = s['idx']
    m = results[tid]
    tile = m['tile']
    
    # VIS cutout
    vx, vy = m['vis_psf_xy'][idx]
    cut_v, lx, ly = _cutout(tile['vis_img'], vx, vy, r=15)
    ax = axes[row, 0]
    ax.imshow(_display_norm(cut_v), cmap='gray_r', origin='lower')
    ax.plot(lx, ly, 'c+', ms=10, mew=2)
    ax.set_title(f'VIS — resid={s["resid_mag"]:.0f}mas  raw={s["raw_mag"]:.0f}mas\n'
                 f'V_SNR={s["vis_snr"]:.0f}  R_SNR={s["rubin_snr"]:.0f}', fontsize=7)
    ax.set_xlim(5, 26); ax.set_ylim(5, 26)
    ax.tick_params(labelsize=5)
    
    # Rubin cutout
    rx, ry = m['rubin_psf_xy'][idx]
    cut_r, lrx, lry = _cutout(m['rubin_target'], rx, ry, r=15)
    ax = axes[row, 1]
    ax.imshow(_display_norm(cut_r), cmap='gray_r', origin='lower')
    ax.plot(lrx, lry, 'c+', ms=10, mew=2)
    ax.set_title(f'Rubin {m["target_band"]} — {tid[-20:]}', fontsize=7)
    ax.set_xlim(5, 26); ax.set_ylim(5, 26)
    ax.tick_params(labelsize=5)

fig.suptitle('Top 10 worst residual offsets\n(VIS left, Rubin right, cyan cross = PSF-fit centroid)',
             fontsize=10, y=1.01)
plt.tight_layout()
plt.show()